# Retail Shelf AI - Train YOLO in Colab (Phases 3-7)
**Before running:** `Runtime > Change runtime type > T4 GPU`.
Upload `retail_shelf_ai_with_data.zip` to the root of your Google Drive (or use the upload option in step 2).

## 1. Check GPU and install

In [ ]:
!nvidia-smi
!pip -q install ultralytics

## 2. Load the project
Option A (recommended): from Google Drive. Option B: upload from your computer.

In [ ]:
USE_DRIVE = True          # set False to upload the zip from your computer instead
ZIP_NAME  = "retail_shelf_ai_with_data.zip"

import os, zipfile
os.chdir("/content")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = f"/content/drive/MyDrive/{ZIP_NAME}"
else:
    from google.colab import files
    up = files.upload()
    zip_path = "/content/" + list(up.keys())[0]

with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")
os.chdir("/content/retail_shelf_ai")
print("Working dir:", os.getcwd())
!ls

## 3. Check classes and labels (Phase 2)

In [ ]:
!cat config.yaml
!python phase2_check_labels.py --preview 3

## 4. Build the YOLO dataset (Phase 3)

In [ ]:
!python phase3_build_dataset.py
!cat dataset/data.yaml

## 5. Train (Phase 4)
~10-15 min for 400 images on a T4. For real products use `yolov8s.pt` and 80-100 epochs.

In [ ]:
EPOCHS = 50
MODEL  = "yolov8n.pt"     # yolov8s.pt = more accurate, a bit slower
!python phase4_train.py --model {MODEL} --epochs {EPOCHS} --imgsz 640 --batch 16 --device 0

## 6. Training curves and validation (Phase 5)

In [ ]:
from IPython.display import Image, display
display(Image("runs/detect/shelf/results.png", width=900))

In [ ]:
!python phase5_validate.py --split test
display(Image("runs/val/test/confusion_matrix.png", width=700))

## 7. Test on held-out images (Phase 6)
Webcam mode needs a local PC (Colab has no camera), so here we run on image files.

In [ ]:
import glob
from IPython.display import Image, display
imgs = sorted(glob.glob("data/heldout/images/*.jpg"))[:3]
for p in imgs:
    !python phase6_webcam.py --source {p} --no-show
    display(Image("outputs/phase6/result_" + p.split("/")[-1], width=700))

## 8. Shelf and slot mapping (Phase 7)
Fixed camera: 8 slots per shelf, shelf edges x = 40 to 920 px (for the synthetic data).
For your own camera, change `--slots` and `--x-range`.

In [ ]:
import json
for p in imgs:
    !python phase7_shelf_mapping.py --source {p} --slots 8 --x-range 40 920 --no-show
    stem = p.split("/")[-1][:-4]
    display(Image(f"outputs/phase7/{stem}_mapped.jpg", width=700))
print(json.dumps(json.load(open(f"outputs/phase7/{stem}_layout.json"))["shelves"][0]["slots"][:2], indent=2))

## 9. Save the trained model
`best.pt` is your trained model. Keep it on Drive and download it for local webcam use (Phases 6-7 live).

In [ ]:
import shutil
os.makedirs("/content/drive/MyDrive/retail_shelf_models", exist_ok=True) if USE_DRIVE else None
if USE_DRIVE:
    shutil.copy("runs/detect/shelf/weights/best.pt", "/content/drive/MyDrive/retail_shelf_models/best.pt")
    print("Saved to Drive: retail_shelf_models/best.pt")
from google.colab import files
files.download("runs/detect/shelf/weights/best.pt")

## Using your own real photos instead
1. Replace `data/raw/images` and `data/raw/labels` in the zip with your annotated YOLO export, and edit `classes:` in `config.yaml` (same order as your labeling tool).
2. Re-zip, upload to Drive, rerun from step 2 with `MODEL = "yolov8s.pt"` and `EPOCHS = 100`.

## Run the trained model locally on a webcam
Copy `best.pt` to `runs/detect/shelf/weights/best.pt` in the project, then:
`python phase6_webcam.py` and `python phase7_shelf_mapping.py --source 0 --slots 8 --x-range 40 920`